In [1]:
%pip install ib_insync

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------- ----------------------------- 3.1/12.3 MB 15.0 MB/s eta 0:00:01
   ---------------- ----------------------- 5.0/12.3 MB 13.0 MB/s eta 0:00:01
   ---------------------- ----------------- 7.1/12.3 MB 11.2 MB/s eta 0:00:01
   ---------------------------- ----------- 8.7/12.3 MB 10.3 MB/s eta 0:00:01
   ---------------------------------- ----- 10.5/12.3 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 9.5 MB/s  0:00:01

   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from ib_insync import IB

ib = IB()
ib.connect('127.0.0.1', 4002, clientId=3)

print("Connecté :", ib.isConnected())
print("Account ID :", ib.managedAccounts())

#ib.disconnect()

In [2]:
import math
import numpy as np

In [ ]:
def norm_cdf(x):
    """
    cumulative distribution function (intégrale de la PDF)
    CDF de la loi normale standard N(x).
    Implémentée from scratch via math.erf (stdlib Python).
    N(x) = 0.5 * (1 + erf(x / sqrt(2)))
    """
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

In [ ]:
def norm_pdf(x):
    """
    Probability Density Function (gaussienne)
    PDF de la loi normale standard n(x).
    n(x) = exp(-x²/2) / sqrt(2π)
    """
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

In [ ]:
def forward(S, r, q, T):
    """
    Forward carry-based (Eq. 3 du roadmap).
    F(T) = S * exp((r - q) * T)
 
    S : spot de référence (mid bid/ask)
    r : taux sans risque continu
    q : dividend yield ou carry implicite
    T : maturité en fractions d'année
    """
    return S * math.exp((r - q) * T)

In [ ]:
def log_moneyness(K, F):
    """
    Log-moneyness relatif au forward (Eq. 6).
    k = ln(K / F)
 
    k < 0 : OTM put / ITM call
    k = 0 : ATM forward
    k > 0 : OTM call / ITM put
    """
    return math.log(K / F)

In [ ]:
def total_variance(sigma, T):
    """
    Variance totale (Eq. 7).
    w = sigma² * T
    """
    return sigma ** 2 * T

In [ ]:
def compute_d1(S, K, T, r, q, sigma):
    """
    d1 (Eq. 8) : [ln(S/K) + (r - q + σ²/2) * T] / (σ * √T)
    d2 (Eq. 9) : d1 - σ * √T
    """
    sqrt_T = math.sqrt(T)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrt_T)
    return d1

In [ ]:
def compute_d2(d1, sigma):
    """
    d1 (Eq. 8) : [ln(S/K) + (r - q + σ²/2) * T] / (σ * √T)
    d2 (Eq. 9) : d1 - σ * √T
    """
    sqrt_T = math.sqrt(T)
    d2 = d1 - sigma * sqrt_T
    return d2

In [ ]:
def bs_call(S, K, T, r, q, sigma):
    """
    Prix call européen Black-Scholes (Eq. 10).
    C = S*e^(-qT)*N(d1) - K*e^(-rT)*N(d2)
    """
    d1 = compute_d1(S, K, T, r, q, sigma)
    d2 = compute_d2(d1, sigma)
    return S * math.exp(-q * T) * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)

In [ ]:
def bs_put(S, K, T, r, q, sigma):
    """
    Prix put européen Black-Scholes (Eq. 11).
    P = K*e^(-rT)*N(-d2) - S*e^(-qT)*N(-d1)
    """
    d1 = compute_d1(S, K, T, r, q, sigma)
    d2 = compute_d2(d1, sigma)
    return K * math.exp(-r * T) * norm_cdf(-d2) - S * math.exp(-q * T) * norm_cdf(-d1)

In [ ]:
def bs_price(S, K, T, r, q, sigma, right):
    """
    Prix Black-Scholes pour call ('C') ou put ('P').
    """
    if right == 'C':
        return bs_call(S, K, T, r, q, sigma)
    elif right == 'P':
        return bs_put(S, K, T, r, q, sigma)
    else:
        raise ValueError(f"right doit être 'C' ou 'P', reçu : {right}")

In [ ]:
def delta(S, K, T, r, q, sigma, right):
    """
    Delta (Eq. 13) : première dérivée du prix par rapport au spot.
    Call : e^(-qT) * N(d1)
    Put  : -e^(-qT) * N(-d1)
    """
    d1 = compute_d1(S, K, T, r, q, sigma)
    d2 = compute_d2(d1, sigma)
    if right == 'C':
        return math.exp(-q * T) * norm_cdf(d1)
    else:
        return -math.exp(-q * T) * norm_cdf(-d1)
 

In [ ]:
def gamma(S, K, T, r, q, sigma):
    """
    Gamma (Eq. 14) : deuxième dérivée par rapport au spot.
    Γ = e^(-qT) * n(d1) / (S * σ * √T)
    Identique pour call et put.
    """
    d1 = compute_d1(S, K, T, r, q, sigma)
    return math.exp(-q * T) * norm_pdf(d1) / (S * sigma * math.sqrt(T))

In [ ]:
def vega(S, K, T, r, q, sigma):
    """
    Vega (Eq. 15) : sensibilité à la volatilité.
    V = S * e^(-qT) * n(d1) * √T
 
    Exprimé par 1 point de vol absolu (0.01 = 1%).
    Identique pour call et put.
    """
    d1 = compute_d1(S, K, T, r, q, sigma)
    vega_raw = S * math.exp(-q * T) * norm_pdf(d1) * math.sqrt(T)
    return vega_raw / 100.0 

In [ ]:
def theta(S, K, T, r, q, sigma, right):
    """
    Theta (Eq. 16) : décroissance temporelle par jour calendaire.
    Exprimé par jour (divisé par 365).
 
    Call : -(S*e^(-qT)*n(d1)*σ)/(2√T) - r*K*e^(-rT)*N(d2) + q*S*e^(-qT)*N(d1)
    Put  : -(S*e^(-qT)*n(d1)*σ)/(2√T) + r*K*e^(-rT)*N(-d2) - q*S*e^(-qT)*N(-d1)
    """
    d1= compute_d1(S, K, T, r, q, sigma)
    d2 = compute_d2(d1, sigma)
    disc_r = math.exp(-r * T)
    disc_q = math.exp(-q * T)
    sqrt_T = math.sqrt(T)
 
    common = -S * disc_q * norm_pdf(d1) * sigma / (2.0 * sqrt_T)
 
    if right == 'C':
        theta_annual = common - r * K * disc_r * norm_cdf(d2) + q * S * disc_q * norm_cdf(d1)
    else:
        theta_annual = common + r * K * disc_r * norm_cdf(-d2) - q * S * disc_q * norm_cdf(-d1)
 
    return theta_annual / 365.0   # par jour calendaire

In [ ]:
def dollar_gamma(S, K, T, r, q, sigma, multiplier=1.0):
    """
    Dollar Gamma (Eq. 17) : gamma monétisé.
    DollarGamma = Γ * S² * multiplier
 
    Pour SX5E options Eurex : multiplier = 10
    """
    g = gamma(S, K, T, r, q, sigma)
    return g * S**2 * multiplier

In [ ]:
def dollar_vega(S, K, T, r, q, sigma, multiplier=1.0):
    """
    Dollar Vega (Eq. 18) : vega monétisé.
    DollarVega = Vega * multiplier
    """
    v = vega(S, K, T, r, q, sigma)
    return v * multiplier

In [ ]:
def pnl_attribution(S, K, T, r, q, sigma, right,
                    dS, d_sigma, dt_days,
                    S_new=None, sigma_new=None, T_new=None):
    """
    Attribution P&L locale (Eq. 19) :
    ΔV ≈ Δ·dS + ½·Γ·(dS)² + Vega·dσ + Θ·dt
 
    Paramètres
    ----------
    dS      : variation du spot
    d_sigma : variation de vol en absolu (ex: 0.01 = +1 vol point)
    dt_days : temps écoulé en jours calendaires
 
    S_new, sigma_new, T_new : si fournis, calcule le résidu
                               vs full revalorisation BS.
 
    Retourne un dict avec chaque composante.
    """
    d = delta(S, K, T, r, q, sigma, right)
    g = gamma(S, K, T, r, q, sigma)
    v = vega(S, K, T, r, q, sigma)
    th = theta(S, K, T, r, q, sigma, right)
 
    delta_pnl = d * dS
    gamma_pnl = 0.5 * g * dS**2
    vega_pnl  = v * d_sigma * 100.0   # vega est par 1%, d_sigma en absolu
    theta_pnl = th * dt_days
    total     = delta_pnl + gamma_pnl + vega_pnl + theta_pnl
 
    result = {
        "delta_pnl" : round(delta_pnl, 4),
        "gamma_pnl" : round(gamma_pnl, 4),
        "vega_pnl"  : round(vega_pnl,  4),
        "theta_pnl" : round(theta_pnl, 4),
        "total_approx" : round(total, 4),
    }
 
    # Résidu vs full revalorisation si état final fourni
    if S_new is not None and sigma_new is not None and T_new is not None:
        price_old = bs_price(S, K, T, r, q, sigma, right)
        price_new = bs_price(S_new, K, T_new, r, q, sigma_new, right)
        full_pnl  = price_new - price_old
        result["full_pnl"] = round(full_pnl, 4)
        result["residual"] = round(full_pnl - total, 4)
 
    return result

In [ ]:
def check_put_call_parity(C, P, S, K, T, r, q, tol=0.01):
    """
    C - P = S*e^(-qT) - K*e^(-rT)
    Retourne True si la parité est respectée dans la tolérance.
    """
    lhs = C - P
    rhs = S * math.exp(-q * T) - K * math.exp(-r * T)
    residual = abs(lhs - rhs)
    print(f"C - P         = {lhs:.4f}")
    print(f"S·e⁻ᵠᵀ-K·e⁻ʳᵀ = {rhs:.4f}")
    print(f"Résidu        = {residual:.6f}  {'✅' if residual <= tol else '❌'}")
    return residual <= tol

In [ ]:
if __name__ == "__main__":
    S, K, T, r, q, sigma = 5000, 5000, 0.25, 0.03, 0.025, 0.18
 
    C = bs_call(S, K, T, r, q, sigma)
    P = bs_put(S, K, T, r, q, sigma)
 
    print(f"Call ATM SX5E : {C:.4f}")
    print(f"Put  ATM SX5E : {P:.4f}")
    print(f"Delta call    : {delta(S, K, T, r, q, sigma, 'C'):.4f}")
    print(f"Gamma         : {gamma(S, K, T, r, q, sigma):.6f}")
    print(f"Vega (1 pt)   : {vega(S, K, T, r, q, sigma):.4f}")
    print(f"Theta (1 jour): {theta(S, K, T, r, q, sigma, 'C'):.4f}")
    print()
    check_put_call_parity(C, P, S, K, T, r, q)
    print()
    attr = pnl_attribution(
        S, K, T, r, q, sigma, 'C',
        dS=50, d_sigma=0.01, dt_days=1,
        S_new=5050, sigma_new=0.19, T_new=T - 1/365
    )
    for k, v_ in attr.items():
        print(f"{k:15s} : {v_}")